<h1 class="alert alert-block alert-info">Create Input Target pair</h1>
<img src="../Images/input-output-pair.png" width="500">

In [1]:
import tiktoken
tokenizer = tiktoken.encoding_for_model("gpt2")

<div class="alert alert-info">Our training data set has <span style="color:orange;">5145 tokens</span> generated using OpenAI tiktoken library (BPE tokenizer)</div>

In [2]:
with open("../Data/the-verdict.txt", mode='r') as file:
    raw_text = file.read()
encoded_text = tokenizer.encode(raw_text)
print(len(encoded_text))

5145


In [3]:
print(encoded_text[:50])

[40, 367, 2885, 1464, 1807, 3619, 402, 271, 10899, 2138, 257, 7026, 15632, 438, 2016, 257, 922, 5891, 1576, 438, 568, 340, 373, 645, 1049, 5975, 284, 502, 284, 3285, 326, 11, 287, 262, 6001, 286, 465, 13476, 11, 339, 550, 5710, 465, 12036, 11, 6405, 257, 5527, 27075, 11]


<div class="alert alert-info">

1. Context size = number of tokens can be fed into the LLM at once
2. Stride = Whats the gap between the first tokens of two context windows.
<div>

<img src="../Images/Input-Target-Mapping.png" width="500">

In [4]:
context_size = 5
print(f"first 5 tokens are {encoded_text[:5]}")
for i in range(1, context_size+1):
    input_tokens = encoded_text[:i]
    target_tokens = encoded_text[i]

    print(f"{input_tokens} --> {target_tokens}")

print(f"\n\nfirst 5 words are {tokenizer.decode(encoded_text[:5])}")
for i in range(1, context_size+1):
    input_tokens = encoded_text[:i]
    target_tokens = encoded_text[i]

    print(tokenizer.decode(encoded_text[:i]), "-->", tokenizer.decode([encoded_text[i]]))

first 5 tokens are [40, 367, 2885, 1464, 1807]
[40] --> 367
[40, 367] --> 2885
[40, 367, 2885] --> 1464
[40, 367, 2885, 1464] --> 1807
[40, 367, 2885, 1464, 1807] --> 3619


first 5 words are I HAD always thought
I -->  H
I H --> AD
I HAD -->  always
I HAD always -->  thought
I HAD always thought -->  Jack


In [5]:
from torch.utils.data import Dataset, DataLoader
import torch

In [6]:
class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, context_size, stride = 0):
        self.input_ids = []
        self.target_ids = []

        # if stride isn't given, then set it equal to the context size
        if stride:
            stride = context_size

        # Get the token ids of input text "txt"
        token_ids = tokenizer.encode(txt)

        for i in range(0, len(token_ids)-context_size, stride):
            input_token = token_ids[i: i+context_size]
            target_token = token_ids[i+1: i+1+context_size]

            # Decoding the tokens to see the input and target tokens in English.
            # print(tokenizer.decode(input_token), "-->", tokenizer.decode(target_token))

            self.input_ids.append(torch.tensor(input_token))
            self.target_ids.append(torch.tensor(target_token))
    def __len__(self):
        return len(self.input_ids)
    
    def __getitem__(self, index):
        return self.input_ids[index], self.target_ids[index]

ds = GPTDatasetV1(raw_text, tokenizer, 4, 4)

In [7]:
def create_datalaoder_v1(txt, batch_size=4, max_length=256, stride=128, shuffle=True, drop_last = True, num_workers=0):
    # initializing the tokenizer
    tokenizer = tiktoken.get_encoding("gpt2")

    # Create dataset 
    dataset = GPTDatasetV1(txt=txt, 
                           tokenizer=tokenizer, 
                           context_size=max_length,
                           stride=stride)
    
    # Create data loader
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers
    )

<img src="../Images/input-output-batch-size-tensor.png" width="500">

In [8]:
with open("../Data/the-verdict.txt", mode='r') as file:
    raw_text = file.read()

max_length = 4
dataloader = create_datalaoder_v1(raw_text, batch_size=8, max_length=max_length, stride=max_length, shuffle=False)

data_itr = iter(dataloader)
first_batch_input, first_batch_target = next(data_itr)
print(f"First Batch: Input tensor: \n{first_batch_input} \n {first_batch_input.shape}")
print(f"First Batch: Target tensor: \n{first_batch_target} \n {first_batch_target.shape}")

First Batch: Input tensor: 
tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]]) 
 torch.Size([8, 4])
First Batch: Target tensor: 
tensor([[  367,  2885,  1464,  1807],
        [ 3619,   402,   271, 10899],
        [ 2138,   257,  7026, 15632],
        [  438,  2016,   257,   922],
        [ 5891,  1576,   438,   568],
        [  340,   373,   645,  1049],
        [ 5975,   284,   502,   284],
        [ 3285,   326,    11,   287]]) 
 torch.Size([8, 4])


<h2 class="alert alert-info">Create Token Embedding layer</h2>
<div class="alert alert-block alert-info">

1. Vocabulary size = 50257
2. Output dimension OR embedding dimension of each token in vocabulary = 256
</div>

<img src="../Images/token-embedding-layer.png" width="500">

In [9]:
context_length = 50257
output_dimension = 256
# creating an embedding of 4 words and 3 dimensions of each word
embedding_layer = torch.nn.Embedding(num_embeddings=context_length, embedding_dim=output_dimension)
print(embedding_layer.weight)

Parameter containing:
tensor([[ 0.2856,  0.6600,  0.6308,  ...,  0.6335, -1.3189,  1.4949],
        [-0.9794, -1.3368,  0.7008,  ...,  0.5371, -0.5480,  0.3735],
        [-1.4816,  1.1132,  1.6219,  ...,  1.2558,  0.4588, -0.8156],
        ...,
        [-1.2001, -1.5532,  0.1934,  ..., -1.0665, -0.2780, -1.1595],
        [ 0.3512, -0.9237, -0.8354,  ...,  1.7532,  0.6502, -1.0388],
        [-0.9194,  1.4132, -0.3637,  ..., -0.0809, -0.5276, -0.3016]],
       requires_grad=True)


<div class="alert alert-block alert-info">

1. First create embedding layer for complete vocabulary (which is done above)
2. Create Dataloader using dataset (done above)
3. Data loader creates batches
   1. Batch size = 8
   2. Context window = 256 (4 in the below pic for demonstration only)
4. Get the embedding of batch size
   1. Leverage embedding layer which act as caching
   2. Output torch.Size([8, 4, 256])
      1. Batch = 8
      2. Context window OR number of tokens = 4
      3. Dimension OR output dimension OR vector dimension OR token Dimension= 256
</div>

<img src="../Images/token-embedding-first-batch.png" width="500">

In [10]:
x = embedding_layer(first_batch_input)
print(x.shape)

torch.Size([8, 4, 256])


<h2 class="alert alert-block alert-info">Positional Embedding</h2>
<img src="../Images/positional-embedding.png" width="500">

In [11]:
# number of ids in each batch = context window
positional_embedding_layer = torch.nn.Embedding(max_length, embedding_dim=output_dimension)
print(torch.arange(max_length))
positional_embedding = positional_embedding_layer(torch.arange(max_length))
print(positional_embedding)
print(positional_embedding.shape)

tensor([0, 1, 2, 3])
tensor([[-1.0472, -0.3695,  0.8088,  ..., -0.7423, -1.1567, -1.0184],
        [-1.6983,  0.3892,  1.7820,  ..., -1.1265, -1.6083, -0.3563],
        [ 1.4882, -0.1048,  0.4631,  ...,  0.8589, -1.2864,  1.6170],
        [ 0.8674,  0.9121, -0.1451,  ..., -1.1391,  0.7256, -1.5265]],
       grad_fn=<EmbeddingBackward0>)
torch.Size([4, 256])


In [12]:
print(embedding_layer(first_batch_input))

tensor([[[ 0.6653, -0.5340, -0.4216,  ...,  0.6217, -0.2282, -1.3473],
         [ 0.6407, -0.0281,  1.7539,  ..., -0.5381,  0.1092,  1.0595],
         [-1.5214,  0.0709,  0.5297,  ...,  0.9963, -0.1167,  0.3271],
         [-1.1748, -1.4395,  0.7871,  ...,  0.9051, -0.1464, -0.2778]],

        [[ 1.1569, -0.9902, -1.0223,  ...,  1.5951,  0.5795, -0.3899],
         [-0.5624, -0.5014,  0.1130,  ..., -0.3323,  1.5958,  0.3057],
         [ 1.2294,  1.3026,  0.9792,  ...,  0.5946,  1.1728, -0.6436],
         [-0.1442, -0.7354, -0.1242,  ..., -0.3927, -1.0629,  0.4099]],

        [[-0.1304, -0.2897, -0.9725,  ..., -0.3320, -0.4792, -0.0551],
         [-0.0309,  0.1530,  0.1162,  ...,  0.8775,  0.8965,  0.3947],
         [ 1.1512, -0.3248,  1.5887,  ...,  0.7841,  1.4470,  1.2501],
         [ 0.0214, -2.0176,  1.1982,  ...,  0.8042,  0.4638,  0.5583]],

        ...,

        [[ 0.3753, -0.7770, -0.7907,  ...,  1.9411,  1.3817, -0.3011],
         [ 0.2112, -0.5131,  0.2259,  ..., -0.3371,  0.13

<h2 class="alert alert-block alert-info">Input Embedding</h2>
<img src="../Images/input-embedding.png" width="500">

In [13]:
#print(embedding_layer(first_batch_input))
input_embedding = embedding_layer(first_batch_input) + positional_embedding
print("Input embedding\n",input_embedding)
print(input_embedding.shape)

Input embedding
 tensor([[[-0.3819, -0.9035,  0.3872,  ..., -0.1205, -1.3849, -2.3657],
         [-1.0576,  0.3611,  3.5359,  ..., -1.6646, -1.4990,  0.7032],
         [-0.0331, -0.0339,  0.9928,  ...,  1.8552, -1.4031,  1.9441],
         [-0.3074, -0.5273,  0.6420,  ..., -0.2340,  0.5792, -1.8042]],

        [[ 0.1096, -1.3597, -0.2135,  ...,  0.8528, -0.5772, -1.4083],
         [-2.2606, -0.1122,  1.8950,  ..., -1.4588, -0.0125, -0.0506],
         [ 2.7176,  1.1978,  1.4423,  ...,  1.4535, -0.1136,  0.9734],
         [ 0.7232,  0.1768, -0.2693,  ..., -1.5318, -0.3374, -1.1165]],

        [[-1.1776, -0.6593, -0.1637,  ..., -1.0743, -1.6359, -1.0735],
         [-1.7292,  0.5422,  1.8982,  ..., -0.2490, -0.7117,  0.0384],
         [ 2.6394, -0.4296,  2.0518,  ...,  1.6430,  0.1606,  2.8671],
         [ 0.8888, -1.1055,  1.0531,  ..., -0.3349,  1.1894, -0.9682]],

        ...,

        [[-0.6719, -1.1465,  0.0181,  ...,  1.1989,  0.2250, -1.3194],
         [-1.4871, -0.1238,  2.0079,  ..